In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing import TypedDict
import subprocess
from openai import OpenAI
import textwrap
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]

In [ ]:


def extract_audio(state: State):
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        "-y",
        output_file
    ]
    subprocess.run(command)
    return  {
        "audio_file": output_file
    }

def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            prompt="프랑스 대혁명에 관련된 내용이야"
        )
        return {
            "transcription": transcription
        }

def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id": i+1, "chunk": chunk})
    
    return [Send("summarize_chunk", chunk) for chunk in chunks]


def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        다음의 텍스트를 요약하세요.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return  {
        "summaries": [summary],
    }


    

In [11]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges("transcribe_audio", dispatch_summarizers, ["summarize_chunk"])
graph_builder.add_edge("summarize_chunk", END)

graph = graph_builder.compile()


In [12]:
graph.invoke({"video_file": "france.mp4"})

Summarizing chunk id 1 chunk: 잠시 후 여러분은 유럽사회를 송두리째 바꿔놓은 프랑스 대혁명에 대해 모두 알 수 있습니다. 자 그럼 로빈의 기록을 시작합니다. 혁명 이전의 프랑스 사회체제는 절대왕정, 신분제를 바 



Summarizing chunk id 2 chunk: 대표들은 표결 방식을 놓고 대립하였습니다. 제1신분과 제2신분은 기존의 신분별 표결을 고집하였지만 제3신분은 전체 회의에 근거한 머릿속 표결과 자신들의 대표 수 증가를 요구하였습니 



Summarizing chunk id 3 chunk: 상황에서 국민의회는 1791년 입헌군주제와 재산에 따른 제한 선거제에 기초한 입헌법에 따라 의회가 해산되고 입법의회가 소집되었습니다. 한편, 혁명의 전파를 우려한 오스트리아와 프로 



Summarizing chunk id 4 chunk: 제거하고 정권을 장악하였습니다. 자코뱅파는 1793년 공화제와 보통선거제에 기조한 헌법을 제정하였지만 전쟁이 끝날 때까지 시행을 유보하였습니다. 또한 봉건적 공납을 무상으로 폐지하 





{'video_file': 'france.mp4',
 'audio_file': 'france.mp3',
 'transcription': '잠시 후 여러분은 유럽사회를 송두리째 바꿔놓은 프랑스 대혁명에 대해 모두 알 수 있습니다. 자 그럼 로빈의 기록을 시작합니다. 혁명 이전의 프랑스 사회체제는 절대왕정, 신분제를 바탕으로 유지되는 체제였습니다. 이 체제를 가리켜 구제도라 읽혔습니다. 소수에 불과한 제1신분인 성직자와 제2신분인 귀족은 절대왕정의 보호 아래 특권층으로 군림하며 많은 토지와 고위관직을 차지하였고 면세특권을 놓였습니다. 하지만 농민, 수공업자, 상인 등 제3신분인 평민은 공건적 의무와 막대한 세금을 부담하면서도 정치적 권리에는 제한을 받는 구제도의 모순 하에 놓여있었습니다. 이에 제3신분 중 상공업과 전문직을 통해 부를 축적한 시민대급은 개몽사상과 미국혁명의 영향을 받아 구제도를 비판하고 자유와 권리를 누리고자 하였습니다. 이 무렵 프랑스는 계속된 전쟁과 왕실의 사치로 재정이 매우 어려워졌습니다. 1789년, 국왕인 루이 16세는 재정 곤란을 해소하기 위해 3부회를 소집하였습니다. 3부회에 모인 3신분의 대표들은 표결 방식을 놓고 대립하였습니다. 제1신분과 제2신분은 기존의 신분별 표결을 고집하였지만 제3신분은 전체 회의에 근거한 머릿속 표결과 자신들의 대표 수 증가를 요구하였습니다. 이것이 받아들여지지 않자 제3신분 대표는 독자적으로 국민의회를 구성하고 헌법이 제정되기 전에는 해상하지 않겠다고 결의하는 테니스코트의 사약을 선언하여 단결을 공고히 하였습니다. 그럼에도 루이 16세가 국민의회를 탄압하자 1789년, 파이 시민들은 절대왕정의 상징인 바스티우 과목을 습격하였습니다. 혁명의 불길은 전국으로 퍼져 농민들은 영주의 성을 습격하였고 장원 문서를 불태웠습니다. 국민의회는 민심을 달래기 위해 봉건제 폐지를 선언하고 자유와 평등, 국민주권, 재산권 보호 등 혁명의 기본 이념을 담은 인간과 시민의 권리 선언 즉, 인권 선언을 발표하여 혁명의 기본 원칙을

In [13]:
transcription = "잠시 후 여러분은 유럽사회를 송두리째 바꿔놓은 프랑스 대혁명에 대해 모두 알 수 있습니다. 자 그럼 로빈의 기록을 시작합니다. 혁명 이전의 프랑스 사회체제는 절대왕정, 신분제를 바탕으로 유지되는 체제였습니다. 이 체제를 가리켜 구제도라 읽혔습니다. 소수에 불과한 제1신분인 성직자와 제2신분인 귀족은 절대왕정의 보호 아래 특권층으로 군림하며 많은 토지와 고위관직을 차지하였고 면세특권을 놓였습니다. 하지만 농민, 수공업자, 상인 등 제3신분인 평민은 공건적 의무와 막대한 세금을 부담하면서도 정치적 권리에는 제한을 받는 구제도의 모순 하에 놓여있었습니다. 이에 제3신분 중 상공업과 전문직을 통해 부를 축적한 시민대급은 개몽사상과 미국혁명의 영향을 받아 구제도를 비판하고 자유와 권리를 누리고자 하였습니다. 이 무렵 프랑스는 계속된 전쟁과 왕실의 사치로 재정이 매우 어려워졌습니다. 1789년, 국왕인 루이 16세는 재정 곤란을 해소하기 위해 3부회를 소집하였습니다. 3부회에 모인 3신분의 대표들은 표결 방식을 놓고 대립하였습니다. 제1신분과 제2신분은 기존의 신분별 표결을 고집하였지만 제3신분은 전체 회의에 근거한 머릿속 표결과 자신들의 대표 수 증가를 요구하였습니다. 이것이 받아들여지지 않자 제3신분 대표는 독자적으로 국민의회를 구성하고 헌법이 제정되기 전에는 해상하지 않겠다고 결의하는 테니스코트의 사약을 선언하여 단결을 공고히 하였습니다. 그럼에도 루이 16세가 국민의회를 탄압하자 1789년, 파이 시민들은 절대왕정의 상징인 바스티우 과목을 습격하였습니다. 혁명의 불길은 전국으로 퍼져 농민들은 영주의 성을 습격하였고 장원 문서를 불태웠습니다. 국민의회는 민심을 달래기 위해 봉건제 폐지를 선언하고 자유와 평등, 국민주권, 재산권 보호 등 혁명의 기본 이념을 담은 인간과 시민의 권리 선언 즉, 인권 선언을 발표하여 혁명의 기본 원칙을 제시하였습니다. 하지만 혁명 이념을 받아들일 수 없었던 루이 16세는 국외로 탈출하려다 체포되어 민중의 반감을 자극하였습니다. 이 상황에서 국민의회는 1791년 입헌군주제와 재산에 따른 제한 선거제에 기초한 입헌법에 따라 의회가 해산되고 입법의회가 소집되었습니다. 한편, 혁명의 전파를 우려한 오스트리아와 프로이센이 프랑스를 위협하자 입법의회는 선전포고를 하고 혁명 전쟁을 일으켰습니다. 전쟁으로 인한 물가 상승과 식량 부족에 시달린 파위민중은 왕궁을 습격하여 왕권을 정지시켰습니다. 그리고 드디어 입법의회 대신 국민공회가 들어섰습니다. 국민공회는 공화정을 선포하고 급진파인 자코뱅파의 주도로 루이 16세를 처형하였고 이에 놀란 영국, 오스트리아 등은 동맹을 맺고 프랑스를 공격하였습니다. 점차 개혁이 급진성을 끼면서 원건파인 지론드파와 급진파인 자코뱅파 간의 갈등이 고조되었습니다. 자코뱅파는 중소시민의 이익을 대변하고 현실 위기를 해결하기 위해 통제경제와 강력한 중앙집권을 주장한 급진공화파들의 모임입니다. 국민공회가 계속된 전쟁으로 인한 급심한 경제난과 반란으로 위기에 처하자 자코뱅파는 로베스피에르를 중심으로 원건파를 제거하고 정권을 장악하였습니다. 자코뱅파는 1793년 공화제와 보통선거제에 기조한 헌법을 제정하였지만 전쟁이 끝날 때까지 시행을 유보하였습니다. 또한 봉건적 공납을 무상으로 폐지하고 물가안정을 위해 최고가격제를 도입하였습니다. 그리고 징경제를 통해 국민군을 조직하여 전쟁에 나섰습니다. 자코뱅파가 세운 혁명정부는 공안위원회와 혁명재판소를 통해 수만명에 닿는 반혁명세력을 무자비하게 처형하며 공포정치를 주도하였습니다. 그러나 로베스피에르가 주도한 공포정치는 점차 많은 사람들의 불만을 삭고 결국 그는 온건파가 주도한 테르미도로의 반동으로 실각하여 처형되었습니다."

for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
    print(chunk)
    print("==========")

잠시 후 여러분은 유럽사회를 송두리째 바꿔놓은 프랑스 대혁명에 대해 모두 알 수 있습니다. 자 그럼 로빈의 기록을 시작합니다. 혁명 이전의 프랑스 사회체제는 절대왕정, 신분제를 바탕으로 유지되는 체제였습니다. 이 체제를 가리켜 구제도라 읽혔습니다. 소수에 불과한 제1신분인 성직자와 제2신분인 귀족은 절대왕정의 보호 아래 특권층으로 군림하며 많은 토지와 고위관직을 차지하였고 면세특권을 놓였습니다. 하지만 농민, 수공업자, 상인 등 제3신분인 평민은 공건적 의무와 막대한 세금을 부담하면서도 정치적 권리에는 제한을 받는 구제도의 모순 하에 놓여있었습니다. 이에 제3신분 중 상공업과 전문직을 통해 부를 축적한 시민대급은 개몽사상과 미국혁명의 영향을 받아 구제도를 비판하고 자유와 권리를 누리고자 하였습니다. 이 무렵 프랑스는 계속된 전쟁과 왕실의 사치로 재정이 매우 어려워졌습니다. 1789년, 국왕인 루이 16세는 재정 곤란을 해소하기 위해 3부회를 소집하였습니다. 3부회에 모인 3신분의
대표들은 표결 방식을 놓고 대립하였습니다. 제1신분과 제2신분은 기존의 신분별 표결을 고집하였지만 제3신분은 전체 회의에 근거한 머릿속 표결과 자신들의 대표 수 증가를 요구하였습니다. 이것이 받아들여지지 않자 제3신분 대표는 독자적으로 국민의회를 구성하고 헌법이 제정되기 전에는 해상하지 않겠다고 결의하는 테니스코트의 사약을 선언하여 단결을 공고히 하였습니다. 그럼에도 루이 16세가 국민의회를 탄압하자 1789년, 파이 시민들은 절대왕정의 상징인 바스티우 과목을 습격하였습니다. 혁명의 불길은 전국으로 퍼져 농민들은 영주의 성을 습격하였고 장원 문서를 불태웠습니다. 국민의회는 민심을 달래기 위해 봉건제 폐지를 선언하고 자유와 평등, 국민주권, 재산권 보호 등 혁명의 기본 이념을 담은 인간과 시민의 권리 선언 즉, 인권 선언을 발표하여 혁명의 기본 원칙을 제시하였습니다. 하지만 혁명 이념을 받아들일 수 없었던 루이 16세는 국외로 탈출하려다 체포되어 민중의 반감을 자극하였습니다. 이
상황에